In [1]:
from dataclasses import replace
from pathlib import Path
import sys

import numpy as np
import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "fine_tuning").exists():
    repo_root = repo_root.parent

for p in (repo_root, repo_root / "src"):
    p = str(p.resolve())
    if p not in sys.path:
        sys.path.insert(0, p)
from fine_tuning.random_search_configs import load_random_search_config
from fine_tuning.fine_tune_configs import load_fine_tune_config
from fine_tuning.eval_fine_tune import evaluate_rolling
from baseline.timeseries.tabdpt_tabpfn.tabdpt_hpo_baseline import (
    load_fine_tune_arrays,
    split_fine_tune_data,
    prediction_horizons,
    _select_horizon_splits,
    maybe_normalize_fine_tune_splits,
    _reset_compiler_state,
    _make_tabdpt,
    preprocess_features,
)

# HPO config to reproduce.
hpo_config = "/home/yuyan/tabdpt_mz/fine_tuning/configs/hpo/v3/agriculture_v3_analyzed_time_features_random_search.yaml"
# hpo_config = "/home/yuyan/tabdpt_mz/fine_tuning/configs/hpo/v3/energy_v3_analyzed_time_features_random_search.yaml"
# hpo_config = "/home/yuyan/tabdpt_mz/fine_tuning/configs/hpo/v3/economy_v3_analyzed_time_features_random_search.yaml"
# hpo_config = "/home/yuyan/tabdpt_mz/fine_tuning/configs/hpo/v3/social_v3_analyzed_time_features_random_search copy.yaml"
hpo_config = "/home/yuyan/tabdpt_mz/fine_tuning/configs/hpo/v3/climate_v3_analyzed_time_features_random_search.yaml"
# Match the runner's default feature setup.
include_calendar_features = True
include_seasonal_features = True
normalize = True

# Drop only the leading lag-incomplete rows from the context split.
drop_incomplete_lag_rows = False

# 1) Read HPO config, then resolve the referenced fine-tune config.
rs_cfg = load_random_search_config(hpo_config)
dataset_name = rs_cfg.base_dataset
run_cfg = load_fine_tune_config(rs_cfg.base_config, dataset_name)

# 2) Build the standard chronological splits first.
X, y, text, timestamps, y_level = load_fine_tune_arrays(run_cfg)
data_splits = split_fine_tune_data(run_cfg, X=X, y=y, text=text, timestamps=timestamps, y_level=y_level)

# 3) Optionally trim only the leading context rows that still contain NaN lag features.
n_lag_dropped_from_context = 0
if drop_incomplete_lag_rows:
    valid_context = np.isfinite(data_splits.X_context).all(axis=1)
    if valid_context.any():
        n_lag_dropped_from_context = int(np.flatnonzero(valid_context)[0])
    else:
        n_lag_dropped_from_context = len(valid_context)

    data_splits = replace(
        data_splits,
        X_context=data_splits.X_context[n_lag_dropped_from_context:],
        y_context=data_splits.y_context[n_lag_dropped_from_context:],
        text_context=data_splits.text_context[n_lag_dropped_from_context:],
        y_level_context=None if data_splits.y_level_context is None else data_splits.y_level_context[n_lag_dropped_from_context:],
        ts_context=None if data_splits.ts_context is None else data_splits.ts_context[n_lag_dropped_from_context:],
    )

n_lag_dropped_from_context


0

In [2]:
# 3) Evaluate every max_context value from the HPO search space.
max_context_values = (
    list(rs_cfg.search_space.max_context)
    if rs_cfg.search_space.max_context is not None
    else [run_cfg.tuning.max_context]
)

def last_level(*parts):
    return None if any(part is None for part in parts) else float(np.concatenate(parts)[-1])

# init a dict to store results for each max_context value, which will be averaged at the end
results_by_max_context = {max_context: [] for max_context in max_context_values}

#iterate over each prediction horizon and max_context value, fit a model, evaluate on val and test, and store the results
for horizon in prediction_horizons(run_cfg):
        raw_splits = _select_horizon_splits(
            data_splits,
            horizon=horizon,
            include_calendar_features=include_calendar_features,
            include_seasonal_features=include_seasonal_features,
        )
        # normalized based on context + train to avoid data leakage from val/test
        splits, target_scaler = maybe_normalize_fine_tune_splits(raw_splits, normalize=normalize)

        prepared = {}
        for max_context in max_context_values:
            _reset_compiler_state()
            # get a fresh model instance for each run
            reg = _make_tabdpt(run_cfg)
            reg.fit(splits.X_context, splits.y_context)

            prepared[max_context] = {
                "reg": reg,
                "X_context_proc": preprocess_features(reg, splits.X_context, reduction_mode=None, reduction_payload=None),
                "X_train_proc": preprocess_features(reg, splits.X_train, reduction_mode=None, reduction_payload=None),
                "X_val_proc": preprocess_features(reg, splits.X_val, reduction_mode=None, reduction_payload=None),
                "X_test_proc": preprocess_features(reg, splits.X_test, reduction_mode=None, reduction_payload=None),
                "metrics": {},
            }

        for split_name in ("val", "test"):
            for max_context in max_context_values:
                reg = prepared[max_context]["reg"]
                X_context_proc = prepared[max_context]["X_context_proc"]
                X_train_proc = prepared[max_context]["X_train_proc"]
                X_val_proc = prepared[max_context]["X_val_proc"]
                X_test_proc = prepared[max_context]["X_test_proc"]

                if split_name == "val":
                    X_hist = np.concatenate((X_context_proc, X_train_proc), axis=0)
                    y_hist = np.concatenate((splits.y_context, splits.y_train), axis=0)
                    text_hist = np.concatenate((splits.text_context, splits.text_train), axis=0)
                    X_eval, y_eval, y_eval_real, text_eval = X_val_proc, splits.y_val, raw_splits.y_val, splits.text_val
                    y_eval_level = raw_splits.y_level_val
                    previous_level = last_level(raw_splits.y_level_context, raw_splits.y_level_train)
                else:
                    # test set
                    X_hist = np.concatenate((X_context_proc, X_train_proc, X_val_proc), axis=0)
                    y_hist = np.concatenate((splits.y_context, splits.y_train, splits.y_val), axis=0)
                    text_hist = np.concatenate((splits.text_context, splits.text_train, splits.text_val), axis=0)
                    X_eval, y_eval, y_eval_real, text_eval = X_test_proc, splits.y_test, raw_splits.y_test, splits.text_test
                    y_eval_level = raw_splits.y_level_test
                    previous_level = last_level(
                        raw_splits.y_level_context,
                        raw_splits.y_level_train,
                        raw_splits.y_level_val,
                    )

                normalized_metrics, real_metrics = evaluate_rolling(
                    reg,
                    X_context_proc=X_hist,
                    y_context=y_hist,
                    text_context=text_hist,
                    X_eval_proc=X_eval,
                    y_eval=y_eval,
                    y_eval_real=y_eval_real,
                    text_eval=text_eval,
                    use_text=False,
                    label=f"{split_name} h={horizon} max_context={max_context}",
                    max_context=max_context,
                    target_scaler=target_scaler,
                    horizon=horizon,
                    y_eval_level=y_eval_level,
                    previous_level=previous_level,
                )
                prepared[max_context]["metrics"][split_name] = {
                    "mae": normalized_metrics[0],
                    "real_mae": real_metrics[0],
                }

        for max_context in max_context_values:
            split_metrics = prepared[max_context]["metrics"]
            results_by_max_context[max_context].append({
                "horizon": horizon,
                "val_mae": split_metrics["val"]["mae"],
                "val_real_mae": split_metrics["val"]["real_mae"],
                "test_mae": split_metrics["test"]["mae"],
                "test_real_mae": split_metrics["test"]["real_mae"],
            })

results = [
    {
        "max_context": max_context,
        "val_mae": np.mean([row["val_mae"] for row in results_by_max_context[max_context]]),
        "val_real_mae": np.mean([row["val_real_mae"] for row in results_by_max_context[max_context]]),
        "test_mae": np.mean([row["test_mae"] for row in results_by_max_context[max_context]]),
        "test_real_mae": np.mean([row["test_real_mae"] for row in results_by_max_context[max_context]]),
        "per_horizon": results_by_max_context[max_context],
    }
    for max_context in max_context_values
]

# 4) Minimal tables: aggregate metrics + per-horizon metrics.
agg_rows = []
horizon_rows = []
for r in results:
    agg_rows.append({k: r[k] for k in ("max_context", "val_mae", "val_real_mae", "test_mae", "test_real_mae")})
    for h in r["per_horizon"]:
        horizon_rows.append({"max_context": r["max_context"], **h})

agg_df = pd.DataFrame(agg_rows)
per_horizon_df = pd.DataFrame(horizon_rows)

agg_df, per_horizon_df

val h=1 max_context=16       | MAE: 2.0691[real] 0.2627[normalized] | RMSE: 2.6387[real] 0.3350[normalized] | MAPE: 4.0273%[real] 23.2729%[normalized]
val h=1 max_context=32       | MAE: 1.4586[real] 0.1852[normalized] | RMSE: 1.9984[real] 0.2537[normalized] | MAPE: 2.8177%[real] 16.8553%[normalized]
val h=1 max_context=64       | MAE: 1.9668[real] 0.2497[normalized] | RMSE: 2.2672[real] 0.2878[normalized] | MAPE: 3.7511%[real] 24.5442%[normalized]
val h=1 max_context=128      | MAE: 1.7997[real] 0.2285[normalized] | RMSE: 2.1700[real] 0.2755[normalized] | MAPE: 3.4337%[real] 22.3217%[normalized]
test h=1 max_context=16      | MAE: 1.7084[real] 0.2169[normalized] | RMSE: 2.3275[real] 0.2955[normalized] | MAPE: 3.7963%[real] 11.5295%[normalized]
test h=1 max_context=32      | MAE: 1.6358[real] 0.2077[normalized] | RMSE: 2.3567[real] 0.2992[normalized] | MAPE: 3.6115%[real] 11.4573%[normalized]
test h=1 max_context=64      | MAE: 2.0917[real] 0.2656[normalized] | RMSE: 2.4592[real] 0.312

(   max_context   val_mae  val_real_mae  test_mae  test_real_mae
 0           16  0.456700      3.537815  0.756443       5.784200
 1           32  0.448024      3.456236  0.748425       5.725268
 2           64  0.743685      5.717672  0.924385       7.074930
 3          128  0.802391      6.166792  1.081837       8.289496,
     max_context  horizon   val_mae  val_real_mae  test_mae  test_real_mae
 0            16        1  0.262698      2.069071  0.216902       1.708371
 1            16        2  0.523157      4.155585  0.387056       3.074497
 2            16        3  0.512953      4.061186  0.576965       4.567988
 3            16        4  0.531114      4.125588  0.881286       6.845651
 4            16        5  0.480774      3.632408  1.079087       8.152859
 5            16        6  0.429504      3.183055  1.397361      10.355833
 6            32        1  0.185194      1.458630  0.207685       1.635772
 7            32        2  0.400891      3.184390  0.416818       3.310908